# Evaluation

Three probes on the extraction pipeline:
1. **Mini golden set** — agreement with hand-labels. No API calls; runs now.
2. **LLM-as-judge** — a second model scores fidelity vs the transcript. Needs a key; a few calls.
3. **Consistency (Gemini)** — re-extract the golden-set patients with Gemini, score them
   against the same golden set (3b), and run the same LLM-as-judge rubric on them (3c) --
   compared directly against the manual Sonnet judge pass in
   `data/outputs/llm_judge_manual.json`. Gated behind `RUN_CONSISTENCY` so it doesn't
   fire until you have a `GEMINI_API_KEY` and your rate limits allow it.

All read the extraction cache (`data/cache/`), so run the pipeline first for real numbers.


## Setup

In [ ]:
import os
# This notebook lives in notebooks/ but all paths (data/, src/) are repo-root-
# relative -- hop up one level if launched from inside notebooks/, no-op otherwise.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import os, json, sys
from pathlib import Path
sys.path.insert(0, 'src')
import evaluate, extract
try:
    from dotenv import load_dotenv; load_dotenv('.env')
except ImportError:
    print('!! pip install -r requirements.txt')
HAS_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
HAS_GEMINI = bool(os.getenv('GEMINI_API_KEY') or os.getenv('GOOGLE_API_KEY'))
CACHE = 'data/cache'
interviews = {d['patient_id']: d['interview_transcript']
              for d in json.loads(Path('data/interviews.json').read_text())}
model_recs = evaluate.load_model_records(CACHE)
print(f'anthropic key: {HAS_KEY} | gemini key: {HAS_GEMINI} | cached records: {len(model_recs)}')

## 1 · Mini golden set

Field-level agreement between the model and `HAND_LABELS` (extend that dict in `src/golden_set.py` to ~8-10 patients). Zero API calls.

In [ ]:
_ = evaluate.evaluate(CACHE)

## 2 · LLM-as-judge

A second model rates fidelity (1-5) of each sampled extraction against its transcript and flags hallucinations / misses. Correct uncertainty is not penalised. Uses a few calls.

In [ ]:
SAMPLE_N = 3
if not HAS_KEY:
    print('no anthropic key -> skipping LLM-as-judge')
elif not model_recs:
    print('no cached records -> run the extraction pipeline first')
else:
    sample = [p for p in interviews if p in model_recs][:SAMPLE_N]
    for pid in sample:
        v = evaluate.judge_record(interviews[pid], model_recs[pid].model_dump_json())
        print(f"{pid}: fidelity={v.get('fidelity')}/5 | "
              f"hallucinations={v.get('hallucinations')} | misses={v.get('misses')}")
        print('    ', v.get('comment', ''))

## 3 · Consistency / model comparison — Gemini

Re-extract a sample with Gemini and compare to the Sonnet run; disagreement is a proxy for reliability. **Gated** — flip `RUN_CONSISTENCY = True` and add `GEMINI_API_KEY` to `.env` when your rate limits allow. This doubles as the 'try a different model' piece.

In [ ]:
RUN_CONSISTENCY = False          # <- set True when ready
GEMINI_MODEL = 'gemini/gemini-1.5-flash'
SAMPLE_N = 3
if not RUN_CONSISTENCY:
    print('RUN_CONSISTENCY = False -> skipping (set True + add GEMINI_API_KEY to run)')
elif not HAS_GEMINI:
    print('no gemini key -> add GEMINI_API_KEY to .env')
elif not model_recs:
    print('no cached records -> run the extraction pipeline first')
else:
    sample = [p for p in interviews if p in model_recs][:SAMPLE_N]
    gemini = {pid: extract.extract_one(interviews[pid], pid, model=GEMINI_MODEL) for pid in sample}
    sonnet = {pid: model_recs[pid] for pid in sample}
    _ = evaluate.agreement_between(sonnet, gemini, 'sonnet', 'gemini')

### 3a · Re-extract the golden-set patients with Gemini

Same `RUN_CONSISTENCY` gate as above. Re-extracts the 10 golden-set patients
specifically (not an arbitrary sample) so the results below are directly
comparable to the Sonnet numbers already in this README -- into a separate
cache dir (`data/cache_gemini/`) so it never touches the real `data/cache/`.
Cheaply resumable: re-running skips any patient already extracted.

In [ ]:
GEMINI_CACHE = Path('data/cache_gemini')
golden_ids = []
if not RUN_CONSISTENCY:
    print('RUN_CONSISTENCY = False -> skipping (set True in the cell above to run this)')
elif not HAS_GEMINI:
    print('no gemini key -> add GEMINI_API_KEY to .env')
else:
    from golden_set import HAND_LABELS
    GEMINI_CACHE.mkdir(parents=True, exist_ok=True)
    golden_ids = list(HAND_LABELS.keys())
    for pid in golden_ids:
        out_path = GEMINI_CACHE / f'{pid}.json'
        if out_path.exists():
            print(f'{pid}: already extracted, skipping')
            continue
        rec = extract.extract_one(interviews[pid], pid, model=GEMINI_MODEL)
        out_path.write_text(rec.model_dump_json(indent=2))
        print(f'{pid}: extracted with Gemini')
    print(f'\nDone -- {len(golden_ids)} golden-set patients in {GEMINI_CACHE}/')

### 3b · Score Gemini's extraction against the same golden set

Runs the identical `evaluate.py` scoring (exact/Jaccard/soft, same as the
README's Q1-Q4 table) against Gemini's extractions, then against Sonnet's,
so the two print blocks below are directly comparable line for line.

In [ ]:
if RUN_CONSISTENCY and HAS_GEMINI and golden_ids:
    print('=== Gemini vs golden set ===')
    gemini_eval = evaluate.evaluate(GEMINI_CACHE)
    print('\n=== Sonnet vs golden set (for direct comparison) ===')
    sonnet_eval = evaluate.evaluate(CACHE)

### 3c · LLM-as-judge on Gemini's extractions, vs. the Claude/Sonnet judge pass

`evaluate.judge_record()` always uses Claude as the judge model (`JUDGE_MODEL`
in `evaluate.py`), regardless of which model produced the extraction being
judged -- deliberate, so both sides of this comparison are graded by the same
judge and the difference reflects extraction quality, not judge bias.

The Sonnet side of this comparison already exists: `data/outputs/llm_judge_manual.json`
is Claude (me, in the conversation that built this repo) manually applying this
exact rubric to the Sonnet extractions, since live judge calls were blocked in
that sandbox. This cell runs the *automated* version of the same rubric against
Gemini's extractions and prints both side by side.

In [ ]:
if RUN_CONSISTENCY and HAS_GEMINI and HAS_KEY and golden_ids:
    manual_claude_judge = json.loads(Path('data/outputs/llm_judge_manual.json').read_text())['results']
    gemini_judge = {}
    for pid in golden_ids:
        rec_path = GEMINI_CACHE / f'{pid}.json'
        if not rec_path.exists():
            continue
        v = evaluate.judge_record(interviews[pid], rec_path.read_text())
        gemini_judge[pid] = v
        claude_f = manual_claude_judge.get(pid, {}).get('fidelity')
        print(f"{pid}: gemini fidelity={v.get('fidelity')}/5  |  sonnet (manual) fidelity={claude_f}/5")
        if v.get('hallucinations'):
            print(f'    gemini hallucinations: {v["hallucinations"]}')
        if v.get('misses'):
            print(f'    gemini misses: {v["misses"]}')
    if gemini_judge:
        mean_gemini = sum(v.get('fidelity', 0) for v in gemini_judge.values()) / len(gemini_judge)
        mean_claude = sum(r.get('fidelity', 0) for r in manual_claude_judge.values()) / len(manual_claude_judge)
        print(f"\nMean fidelity -- Gemini extractions: {mean_gemini:.1f}/5  |  Sonnet extractions: {mean_claude:.1f}/5")
        out = Path('data/outputs/llm_judge_gemini.json')
        out.write_text(json.dumps(gemini_judge, indent=2))
        print(f'Wrote {out}')

## Reading the results

- Golden set tells you where the model matches human judgment (and where it doesn't) — the core 'is it working' signal.
- Judge catches fabrication/omission the golden set's fixed fields might miss.
- Consistency shows which fields are stable across models vs model-dependent.

Write up: what's solid, what's shaky, what you'd fix first.